In [1]:
# Initialize Spark with Delta Lake and required packages
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta import configure_spark_with_delta_pip

my_packages = [
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "org.apache.hadoop:hadoop-client-runtime:3.3.4",
    "org.apache.hadoop:hadoop-client-api:3.3.4",
    "io.delta:delta-core_2.12:2.4.0",
    "io.delta:delta-storage:2.4.0",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",
]

builder = pyspark.sql.SparkSession.builder \
    .appName("BankingLakehouse-SmokeTest") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.databricks.delta.retentionDurationCheck.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

spark = configure_spark_with_delta_pip(builder, extra_packages=my_packages).getOrCreate()

print(f"✅ Spark version: {spark.version}")
print(f"✅ Delta Lake configured")

✅ Spark version: 3.4.1
✅ Delta Lake configured


In [2]:
# Create sample banking data
schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("account_id", StringType(), False),
    StructField("balance", IntegerType(), False)
])

data = [
    ("C001", "ACC001", 50000),
    ("C002", "ACC002", 75000),
    ("C003", "ACC003", 120000)
]

df = spark.createDataFrame(data, schema)

In [3]:
# Write to Delta Lake (Bronze layer simulation)
delta_path = "s3a://bronze/smoke_test"
df.write.format("delta").mode("overwrite").save(delta_path)

print(f"✅ Written {df.count()} records to Delta Lake")

✅ Written 3 records to Delta Lake


In [4]:
# Read back and verify
df_read = spark.read.format("delta").load(delta_path)
df_read.show()

# Check Delta transaction log
print("\n📋 Delta Lake Transaction History:")
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, delta_path)
delta_table.history().select("version", "timestamp", "operation").show()

spark.stop()

+-----------+----------+-------+
|customer_id|account_id|balance|
+-----------+----------+-------+
|       C002|    ACC002|  75000|
|       C001|    ACC001|  50000|
|       C003|    ACC003| 120000|
+-----------+----------+-------+


📋 Delta Lake Transaction History:
+-------+-------------------+---------+
|version|          timestamp|operation|
+-------+-------------------+---------+
|      0|2026-02-27 01:45:28|    WRITE|
+-------+-------------------+---------+

